<a href="https://colab.research.google.com/github/kushalshah0/NeuraMind/blob/main/ollama_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Install Zstandard (zstd) Dependency

In [10]:
print('Installing zstd dependency...')
!sudo apt-get update && sudo apt-get install -y zstd
print('zstd installed.')

Installing zstd dependency...
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list e

### 2. Install Ollama

In [11]:
print('Installing Ollama...')
!curl -fsSL https://ollama.com/install.sh | sh
print('Ollama installation script executed. Verifying installation.')

Installing Ollama...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Ollama installation script executed. Verifying installation.


### 3. Install PCI Utilities for GPU Detection

In [12]:
print('Installing pciutils for GPU detection...')
!sudo apt-get install -y pciutils
print('pciutils installed. Now we need to restart Ollama for it to recognize the GPU.')

Installing pciutils for GPU detection...
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pciutils is already the newest version (1:3.7.0-6).
0 upgraded, 0 newly installed, 0 to remove and 119 not upgraded.
pciutils installed. Now we need to restart Ollama for it to recognize the GPU.


### 4. Start Ollama Server

In [13]:
print('Starting Ollama server with GPU detection enabled...')
import os
os.environ['OLLAMA_HOST'] = '0.0.0.0'
os.environ['OLLAMA_ORIGINS'] = '*'
get_ipython().system_raw('ollama serve &') # Start Ollama server in the background
print('Ollama server started. It should now detect GPU if available.')

Starting Ollama server with GPU detection enabled...
Ollama server started. It should now detect GPU if available.


### 5. Pull Code-Generation Model

In [19]:
print('Attempting to pull qwen2.5-coder:14b model...')
import subprocess
import time

# Give Ollama a moment to fully start after setting OLLAMA_HOST
time.sleep(10)

# Attempt to pull the requested model
!ollama pull qwen2.5-coder:14b

# Check if the model was pulled successfully
try:
    result = subprocess.run(['ollama', 'list'], capture_output=True, text=True, check=True)
    ollama_list_output = result.stdout
    print(ollama_list_output)

    if 'qwen2.5-coder:14b' in ollama_list_output:
        print('qwen2.5-coder:14b model found.')
    else:
        print('Error: qwen2.5-coder:14b model not found after pull attempt.')

except subprocess.CalledProcessError as e:
    print(f"Error executing 'ollama list': {e.stderr}")
    print('Could not verify model after pull attempt.')

Attempting to pull qwen2.5-coder:14b model...

NAME                  ID              SIZE      MODIFIED               
qwen2.5-coder:14b     9ec8897f747e    9.0 GB    Less than a second ago    
qwen3-coder:latest    06c1097efce0    18 GB     12 minutes ago            
qwen:latest           d53d04290064    2.3 GB    21 minutes ago            

qwen2.5-coder:14b model found.


### 6. Install Cloudflared

In [15]:
print('Installing cloudflared...')
!curl -L --output cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && sudo dpkg -i cloudflared.deb
print('cloudflared installation initiated.')

Installing cloudflared...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 18.2M  100 18.2M    0     0  22.5M      0 --:--:-- --:--:-- --:--:-- 22.5M
(Reading database ... 121736 files and directories currently installed.)
Preparing to unpack cloudflared.deb ...
Unpacking cloudflared (2026.2.0) over (2026.2.0) ...
Setting up cloudflared (2026.2.0) ...
Processing triggers for man-db (2.10.2-1) ...
cloudflared installation initiated.


### 7. Create Cloudflare Tunnel

In [16]:
import time
import re
import os

print('Creating Cloudflare tunnel for Ollama...')
output_file = 'cloudflared_tunnel_output.txt'

# Run cloudflared tunnel in the background, redirecting output to a file
# Using '2>&1' to redirect stderr to stdout, so all messages including the URL are captured
get_ipython().system_raw(f'cloudflared tunnel --url http://localhost:11434 > {output_file} 2>&1 &')

print('Waiting for tunnel to establish and URL to be generated...')
time.sleep(10) # Give some time for the tunnel to establish and print the URL

public_url = None
if os.path.exists(output_file):
    with open(output_file, 'r') as f:
        output_content = f.read()
    print(f'Cloudflared output:\n{output_content}')

    # Regex to find the URL in the cloudflared output
    # The URL typically starts with 'https://' and ends with '.trycloudflare.com'
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', output_content)
    if match:
        public_url = match.group(0)

if public_url:
    print(f'\n!!! Your Ollama Public URL is: {public_url} !!!')
    os.environ['OLLAMA_PUBLIC_URL'] = public_url # Store for later use
else:
    print('\nError: Could not find the public Cloudflare tunnel URL in the output.')
    print('Please check the output above for any errors or try running the command manually.')

Creating Cloudflare tunnel for Ollama...
Waiting for tunnel to establish and URL to be generated...
Cloudflared output:
2026-02-07T12:28:43Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-02-07T12:28:43Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-02-07T12:28:47Z INF +--------------------------------------------------------------------------------------------+
2026-02-07T12:28:47Z INF |  Your quick Tunnel has been created! Visit it at (it m

### 8. Verify Ollama API via Tunnel

In [20]:
import requests
import json
import os

print('Verifying Ollama API via Cloudflare tunnel...')

public_url = os.environ.get('OLLAMA_PUBLIC_URL')

if not public_url:
    print('Error: OLLAMA_PUBLIC_URL not found in environment variables.')
else:
    print(f'Using public URL: {public_url}')
    api_endpoint = f'{public_url}/api/generate'

    payload = {
        "model": "qwen2.5-coder:14b",
        "prompt": "Write a simple Python function to add two numbers."
    }

    headers = {'Content-Type': 'application/json'}

    try:
        response = requests.post(api_endpoint, headers=headers, data=json.dumps(payload), timeout=120)
        response.raise_for_status() # Raise an exception for HTTP errors

        print(f"Status Code: {response.status_code}")

        full_response_content = ""
        generated_text = ""
        for line in response.iter_lines():
            if line:
                decoded_line = line.decode('utf-8')
                full_response_content += decoded_line + '\n'
                try:
                    json_response = json.loads(decoded_line)
                    if 'response' in json_response:
                        generated_text += json_response['response']
                    if json_response.get('done'):
                        break
                except json.JSONDecodeError:
                    print(f"Could not decode JSON from line: {decoded_line}")

        print("\n--- Full Ollama API Response (parsed line by line) ---")
        print(full_response_content)
        print("-----------------------------------------------------")

        if generated_text:
            print("\n--- Extracted Generated Text ---")
            print(generated_text)
            print("----------------------------------")
        else:
            print("\nNo generated text found in the response.")

    except requests.exceptions.RequestException as e:
        print(f"Error connecting to Ollama API: {e}")
        if hasattr(e, 'response') and e.response is not None:
            print(f"Response content: {e.response.text}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

Verifying Ollama API via Cloudflare tunnel...
Using public URL: https://neither-illustration-bond-observation.trycloudflare.com
Status Code: 200

--- Full Ollama API Response (parsed line by line) ---
{"model":"qwen2.5-coder:14b","created_at":"2026-02-07T12:42:11.412017111Z","response":"Certainly","done":false}
{"model":"qwen2.5-coder:14b","created_at":"2026-02-07T12:42:11.476823482Z","response":"!","done":false}
{"model":"qwen2.5-coder:14b","created_at":"2026-02-07T12:42:11.524492135Z","response":" Below","done":false}
{"model":"qwen2.5-coder:14b","created_at":"2026-02-07T12:42:11.570141176Z","response":" is","done":false}
{"model":"qwen2.5-coder:14b","created_at":"2026-02-07T12:42:11.615937777Z","response":" a","done":false}
{"model":"qwen2.5-coder:14b","created_at":"2026-02-07T12:42:11.66229156Z","response":" simple","done":false}
{"model":"qwen2.5-coder:14b","created_at":"2026-02-07T12:42:11.708688192Z","response":" Python","done":false}
{"model":"qwen2.5-coder:14b","created_at":"2

### 9. Example Ollama API Call (from External)

In [21]:
import requests
import json
import os

print('Executing Ollama API call to the tunneled URL:')

public_url = os.environ.get('OLLAMA_PUBLIC_URL')

if not public_url:
    print('Error: OLLAMA_PUBLIC_URL not found in environment variables. Please ensure the tunnel was successfully created.')
else:
    ollama_api_url = f"{public_url}/api/generate"

    payload = {
        "model": "qwen2.5-coder:14b",
        "prompt": "Write a js code to add two number",
        "stream": True
    }

    headers = {'Content-Type': 'application/json'}

    print(f"Sending request to: {ollama_api_url}")

    try:
        response = requests.post(ollama_api_url, headers=headers, data=json.dumps(payload), stream=True, timeout=120)
        response.raise_for_status() # Raise an exception for HTTP errors (4xx or 5xx)

        print("\nGenerated Code:")
        generated_text = ""
        for line in response.iter_lines():
            if line:
                decoded_line = line.decode('utf-8')
                try:
                    json_response = json.loads(decoded_line)
                    if 'response' in json_response:
                        generated_text += json_response['response']
                    if json_response.get('done'):
                        break
                except json.JSONDecodeError:
                    print(f"Could not decode JSON from line: {decoded_line}")
        print(generated_text)

    except requests.exceptions.RequestException as e:
        print(f"Error calling Ollama API: {e}")
        if hasattr(e, 'response') and e.response is not None:
            print(f"Response content: {e.response.text}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

Executing Ollama API call to the tunneled URL:
Sending request to: https://neither-illustration-bond-observation.trycloudflare.com/api/generate

Generated Code:
Certainly! Below is a simple JavaScript function that takes two numbers as arguments and returns their sum:

```javascript
function addTwoNumbers(num1, num2) {
    return num1 + num2;
}

// Example usage:
const result = addTwoNumbers(5, 3);
console.log(result); // Output: 8
```

You can call the `addTwoNumbers` function with any two numbers to get their sum. The example usage demonstrates adding 5 and 3, which results in 8.
